# Preparing the HE 1002-0755 spectrum
### Worksheet 1's procedure, applied to our own star

In [ ]:
import sys, os, json
sys.path.insert(0, "../../../scripts")
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from astropy.io import fits
from astropy.table import Table
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
import astropy.units as u
from PyAstronomy import pyasl
from mike_multispec import read_multispec, normalise_order, merge_orders

DATA = "../../../data"
BLUE = f"{DATA}/he1002-0755_epochs/he1002-0755ablue_multi_ut090205_07slit.fits"
RED  = f"{DATA}/he1002-0755ared_multi_ut090205_07slit.fits"
C_KMS = 299792.458
os.makedirs("plots", exist_ok=True); os.makedirs("results", exist_ok=True)

## 1. Reading the multispec files

In [ ]:
blue, hdr_b = read_multispec(BLUE)
red,  hdr_r = read_multispec(RED)

for tag, orders, h in [("blue", blue, hdr_b), ("red", red, hdr_r)]:
    snr = [np.nanmedian(o[2]) for o in orders]
    print(f"{tag:5s} {h['INSTRUME']:10s} {len(orders):2d} orders  "
          f"{min(o[0].min() for o in orders):.0f}-{max(o[0].max() for o in orders):.0f} A   "
          f"median S/N per order: max {np.nanmax(snr):.0f}, median {np.nanmedian(snr):.0f}")
    print(f"      UT {h['UT-DATE']} {h['UT-START']}   exp {h['EXPTIME']:.0f}s   "
          f"slit {h['SLITSIZE']}   airmass {h['AIRMASS']}")
    dop = [k for k in h if "DOPCOR" in k]
    dopstr = {k: h[k] for k in dop} or "NONE"
    print(f"      DOPCOR history: {dopstr}")


## 2. Continuum normalisation

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 8))
for ax, (tag, orders, i) in zip(axes, [("blue", blue, 18), ("red", red, 20), ("red", red, 8)]):
    w, f, s = orders[i]
    nf, cont = normalise_order(w, f)
    ax.plot(w, f/np.nanmedian(f), color="0.65", lw=0.7, label="raw / median")
    ax.plot(w, cont/np.nanmedian(f), "r-", lw=1.4, label="continuum fit")
    ax.plot(w, nf, "k-", lw=0.8, label="normalised")
    ax.axhline(1, color="b", ls=":", lw=1)
    ax.set_ylim(0, 1.6)
    ax.set_title(f"{tag} order {i}   {w[0]:.0f}-{w[-1]:.0f} A   median S/N {np.nanmedian(s):.0f}",
                 fontsize=10)
    ax.legend(fontsize=8, ncol=3, loc="lower right")
axes[-1].set_xlabel("Wavelength (A)")
fig.tight_layout(); fig.savefig("plots/continuum_normalisation.png", dpi=130)
plt.show()

## 3. Merging

In [ ]:
all_orders = blue + red
kept = [o for o in all_orders if np.nanmedian(o[2]) >= 8.0]
print(f"{len(kept)} of {len(all_orders)} orders pass the S/N >= 8 cut")

wave, flux = merge_orders(all_orders)
print(f"merged: {len(wave)} points, {wave[0]:.1f} - {wave[-1]:.1f} A, "
      f"median normalised flux {np.nanmedian(flux):.3f}")

fig, ax = plt.subplots(2, 1, figsize=(13, 6))
ax[0].plot(wave, flux, "k-", lw=0.4); ax[0].set_xlim(3830, 9150); ax[0].set_ylim(0, 1.4)
ax[0].set_title("HE 1002-0755, merged and normalised (observed frame)")
m = (wave > 5160) & (wave < 5200)
ax[1].plot(wave[m], flux[m], "k-", lw=1.0); ax[1].set_ylim(0, 1.2)
ax[1].set_title("Mg b region"); ax[1].set_xlabel("Wavelength (A)")
for a in ax: a.set_ylabel("Normalised flux"); a.grid(alpha=0.25)
fig.tight_layout(); fig.savefig("plots/merged_spectrum.png", dpi=130)
plt.show()

## 4. Radial velocity

In [ ]:
def load1d(p):
    hh = fits.open(p)[0]; h = hh.header; n = h["NAXIS1"]
    cd = h.get("CD1_1", h.get("CDELT1"))
    return h["CRVAL1"] + cd*(np.arange(n) + 1 - h.get("CRPIX1", 1)), np.asarray(hh.data, float)

METAL = {"Mg b 5160-5200": (5160, 5200), "Fe 4400-4700": (4400, 4700),
         "Fe 5250-5500": (5250, 5500), "Fe 5500-5800": (5500, 5800),
         "Fe/Cr 4700-5000": (4700, 5000)}
EXCLUDED = {"H-alpha 6560-6575": (6560, 6575)}

def ccf(w, f, wt, ft, lo, hi, vmin=100, vmax=340, pad=25):
    m = (w > lo) & (w < hi); mt = (wt > lo - pad) & (wt < hi + pad)
    if m.sum() < 50 or mt.sum() < 50: return np.nan
    r, c = pyasl.crosscorrRV(w[m], f[m] - np.nanmedian(f[m]),
                             wt[mt], ft[mt] - np.nanmedian(ft[mt]), vmin, vmax, 0.1, skipedge=20)
    return float(r[np.argmax(c)])

results = {}
for tname in ["HD122563", "HD140283", "cs22892-52"]:
    wt, ft = load1d(f"{DATA}/{tname}.fits")
    vs = {k: ccf(wave, flux, wt, ft, *v) for k, v in METAL.items()}
    ha = ccf(wave, flux, wt, ft, *EXCLUDED["H-alpha 6560-6575"])
    good = [v for v in vs.values() if np.isfinite(v)]
    results[tname] = dict(per_window=vs, mean=float(np.mean(good)),
                          scatter=float(np.std(good, ddof=1)), halpha=ha)
    print(f"== {tname}")
    for k, v in vs.items(): print(f"     {k:<18s} {v:+8.2f}")
    print(f"     {'H-alpha':<18s} {ha:+8.2f}   EXCLUDED")
    print(f"     mean {np.mean(good):+8.2f}  scatter {np.std(good, ddof=1):.2f} km/s\n")

V_GEO = results["HD122563"]["mean"]
V_ERR = results["HD122563"]["scatter"]
print(f"adopted v_geo = {V_GEO:+.2f} +/- {V_ERR:.2f} km/s  (HD 122563, the closest analogue)")
print(f"template-to-template spread: "
      f"{max(r['mean'] for r in results.values()) - min(r['mean'] for r in results.values()):.2f} km/s")

## 5. Which frame is the wavelength scale in?

In [ ]:
w140, f140 = load1d(f"{DATA}/HD140283.fits")
V_TELL_HD140283 = 141.92

print("telluric velocity in the HE 1002-0755 merged spectrum:")
tv = []
for name, (lo, hi) in {"H2O 7160-7350": (7160, 7350), "H2O 8100-8350": (8100, 8350),
                       "H2O 8900-9150": (8900, 9150), "O2 B-band 6865-6935": (6865, 6935)}.items():
    d = ccf(wave, flux, w140, f140, lo, hi, -260, -30, pad=40)
    tv.append(d + V_TELL_HD140283)
    print(f"   {name:<22s} delta {d:+8.2f}  ->  {d + V_TELL_HD140283:+6.2f} km/s")
print(f"\n   mean {np.mean(tv):+.2f} km/s  (0 = geocentric)")
print("   => the wavelength scale is GEOCENTRIC; the heliocentric correction must be applied.")

## 6. Heliocentric correction — and the date

In [ ]:
lco = EarthLocation.from_geodetic(lon=hdr_b["SITELONG"]*u.deg, lat=hdr_b["SITELAT"]*u.deg,
                                  height=hdr_b["SITEALT"]*u.m)
sky = SkyCoord(ra=hdr_b["RA-D"]*u.deg, dec=hdr_b["DEC-D"]*u.deg)
GAIA_RV, GAIA_E = 226.676, 3.710

helio = {}
for iso in [f"2009-02-05T{hdr_b['UT-START']}", f"2009-07-27T{hdr_b['UT-START']}"]:
    t = Time(iso, scale="utc", location=lco)
    helio[iso[:10]] = float(sky.radial_velocity_correction("heliocentric",
                                                           obstime=t).to(u.km/u.s).value)
for d, hc in helio.items():
    v = V_GEO + hc
    print(f"   {d}  corr {hc:+7.3f}  ->  v_helio {v:+7.2f}   "
          f"Gaia offset {v - GAIA_RV:+6.2f} km/s ({abs(v - GAIA_RV)/GAIA_E:.1f} sigma)")

HELCORR = helio["2009-02-05"]
V_HELIO = V_GEO + HELCORR
print(f"\n=> adopt the header date. v_helio = {V_HELIO:+.2f} km/s")

## 7. Shift to rest and save

In [ ]:
w_rest = wave/(1.0 + V_GEO/C_KMS)
t = Table([w_rest, flux], names=["wavelength", "norm_flux"])
t.meta["VGEO"] = V_GEO; t.meta["VHELIO"] = V_HELIO; t.meta["HELCORR"] = HELCORR
t.write(f"{DATA}/he1002-0755_rest.fits", overwrite=True)
np.savetxt(f"{DATA}/he1002-0755_rest.txt", np.c_[w_rest, flux], fmt="%.5f %.6f")

summary = dict(v_geo_kms=V_GEO, v_geo_err_kms=V_ERR, helcorr_kms=HELCORR, v_helio_kms=V_HELIO,
               gaia_rv_kms=GAIA_RV, gaia_err_kms=GAIA_E, telluric_kms=float(np.mean(tv)),
               per_template={k: v["mean"] for k, v in results.items()},
               halpha_excluded_kms=results["HD122563"]["halpha"],
               n_orders_used=len(kept), wave_min=float(w_rest[0]), wave_max=float(w_rest[-1]))
json.dump(summary, open("results/prepare_spectrum.json", "w"), indent=2)

fig, ax = plt.subplots(figsize=(12, 4))
m = (w_rest > 5160) & (w_rest < 5200)
ax.plot(w_rest[m], flux[m], "k-", lw=1.1, label="HE 1002-0755, rest frame")
w122, f122 = load1d(f"{DATA}/HD122563.fits")
m2 = (w122 > 5160) & (w122 < 5200)
ax.plot(w122[m2], f122[m2], "b-", lw=0.9, alpha=0.7, label="HD 122563")
ax.set_xlabel("Rest wavelength (A)"); ax.set_ylabel("Normalised flux")
ax.legend(frameon=False); ax.grid(alpha=0.25); ax.set_ylim(0, 1.2)
fig.tight_layout(); fig.savefig("plots/rest_frame_mgb.png", dpi=130)
plt.show()

for k, v in summary.items():
    if isinstance(v, float): print(f"   {k:<24s} {v:+.3f}")
print(f"\nwrote {DATA}/he1002-0755_rest.fits and .txt")

## Caveat that carries forward